In [1]:
import pandas as pd # para manipular dados em formato de tabelas dataframes
import time # usada para medir o tempo calculando o custo computacional
from sklearn.neural_network import MLPClassifier # algoritmo de redes neurais
from sklearn.model_selection import GridSearchCV # para testar as combinações dos hiperparâmetros
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score # funções para extrair a matriz de confusão, o relatório das métricas e a acurácia

Abrindo e carregando os dados dos arquivos iris_treino e iris_teste em .csv, verificando também se há valores ausentes.

In [2]:
treino = pd.read_csv('iris_treino.csv')
teste = pd.read_csv('iris_teste.csv')

# verificação de valores ausentes
print("Valores ausentes no treino:")
print(treino.isnull().sum())

print("\nValores ausentes no teste:")
print(teste.isnull().sum())

Valores ausentes no treino:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
class           0
dtype: int64

Valores ausentes no teste:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
class           0
dtype: int64


Separando as características das espécies da coluna "class" para que o Random Forest possa aprender.

In [3]:
X_treino = treino.drop(columns=['class']) # armazena as medidas das pétalas e sépalas excluindo a coluna 'class' da tabela treino
y_treino = treino['class'] # isola e armazena apenas a coluna 'class' da tabela treino
X_teste = teste.drop(columns=['class']) # armazena as medidas das pétalas e sépalas excluindo a coluna 'class' da tabela teste
y_teste = teste['class'] # isola e armazena apenas a coluna 'class' da tabela teste

O X é maiúsculo porque representa uma matriz matemática que guarda apenas as colunas com as características físicas das flores, como por exemplo o comprimento da sépala, a largura da sépala, o comprimento da pétala e a largura da pétala. Essas são as pistas, que o algoritmo vai usar para tentar adivinhar o tipo de flor.

Já o y, é minúsculo porque representa uma única coluna, onde guarda o "gabarito", as respostas com os nomes das espécies, como por exemplo: Setosa, Versicolor, Virginica, etc.

Em seguida, criamos um dicionário com 'param_classificador' para treinar o classificador Redes Neurais, para definir a arquitetura do cérebro artificial.

In [4]:
# definindo a grade para testar a topologia/arquitetura dos neurônios (Tópico 10)
param_classificador = {
    'hidden_layer_sizes': [(50,), (100,), (50, 25)],  # configuração de camadas ocultas e neurônios
    'max_iter': [1000]                                # máximo de iterações para garantir convergência
}

O 'hidden_layer_sizes' define quantas camadas escondidas de neurônios o modelo terá e quantos neurônios haverá em cada uma: (50,) testa uma arquitetura com 1 única camada oculta contendo 50 neurônios, (100,) testa uma arquitetura com 1 única camada oculta contendo 100 neurônios, e (50, 25) testa uma arquitetura mais profunda, com 2 camadas ocultas, a primeira com 50 neurônios e a segunda com 25 neurônios.

O 'max_iter' é o número máximo de iterações que a rede tem direito de fazer. Definindo em [1000] garante que a rede tenha tempo suficiente para aprender e concentrar-se matematicamente sem interromper o treino no meio.

Logo iniciamos o cronômetro usando a biblioteca time para medir o custo computacional do treinamento do algoritmo.

In [5]:
# iniciando o cronômetro para medir o custo computacional do treino
inicio = time.time()

# treinando o classificador Redes Neurais
classificador = GridSearchCV(MLPClassifier(random_state=42), param_classificador, cv=3)
classificador.fit(X_treino, y_treino)

# calculando o tempo total gasto no treinamento
tempo_execucao = time.time() - inicio

Aqui definimos o algoritmo de Redes Neurais que será treinado com a base de treino.

O random_state=42 está dentro, pois o MLPClassifier ativa a rede neural e trava os pesos iniciais das sinapses, as conexões, na semente 42, para garantir que os números dessas "conexões" sejam preenchidos com valores iniciais diferentes de 0, e que o aprendizado comece sempre do mesmo ponto matemático para manter a reprodutibilidade. O cv=3 é uma validação cruzada em 3 blocos, como temos 3 combinações de arquitetura de neurônios, a máquina rodará 9 treinos, 3 arquiteturas e 3 blocos de teste.

A variável classificador que é o objeto do GridSearchCV. O comando '.best_estimator_' vai lá dentro, pega a rede neural que ganhou com um resultado maior de acurácia média da validação cruzada e salva na variável 'final'.

Esse comando vai printar na tela mostrando as configurações do modelo vencedor. Ele aprende regras de decisão. O print do objeto 'final' mostra os hiperparâmetros escolhidos, como n_estimators e max_depth.

In [6]:
# armazenando o melhor modelo encontrado pelo GridSearchCV
final = classificador.best_estimator_

# visualizando o modelo
print("Modelo Redes Neurais:")
print(final)

Modelo Redes Neurais:
MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, random_state=42)


Por fim, tentamos prever as classes da base de teste e exibimos os resultados. O '.best_params_' busca no objeto qual das 3 arquiteturas de neurônios que venceram a validação cruzada. E usamos 'tempo_execucao' para printar o tempo do treinamento do classificador.

In [23]:
# realizando as previsões na base de teste isolada
previsoes = final.predict(X_teste)

# saída de dados
print(f"Melhores hiperparâmetros encontrados: {classificador.best_params_}")
print(f"Tempo de treino: {tempo_execucao:.5f} segundos")

# a acurácia
acuracia = accuracy_score(y_teste, previsoes)
print(f"Acurácia: {acuracia:.5f}\n")

# matriz de confusão
matriz = confusion_matrix(y_teste, previsoes)
print("----- Matriz de confusão -----")
cm_t = matriz.T
print(f"Precisão \t Iris-setosa \t Iris-versicolor \t Iris-virginica")
print(f"Iris-setosa \t\t {cm_t[0][0]} \t\t {cm_t[0][1]} \t\t\t {cm_t[0][2]}")
print(f"Iris-versicolor \t {cm_t[1][0]} \t\t {cm_t[1][1]} \t\t\t {cm_t[1][2]}")
print(f"Iris-virginica \t\t {cm_t[2][0]} \t\t {cm_t[2][1]} \t\t\t {cm_t[2][2]}")

# relatório geral
print("\n----- Relatório de classificação por classe e média macro -----")
relatorio_texto = classification_report(y_teste, previsoes)
print(relatorio_texto)

Melhores hiperparâmetros encontrados: {'hidden_layer_sizes': (50,), 'max_iter': 1000}
Tempo de treino: 5.27580 segundos
Acurácia: 0.97778

----- Matriz de confusão -----
Precisão 	 Iris-setosa 	 Iris-versicolor 	 Iris-virginica
Iris-setosa 		 15 		 0 			 0
Iris-versicolor 	 0 		 15 			 1
Iris-virginica 		 0 		 0 			 14

----- Relatório de classificação por classe e média macro -----
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        15
Iris-versicolor       0.94      1.00      0.97        15
 Iris-virginica       1.00      0.93      0.97        15

       accuracy                           0.98        45
      macro avg       0.98      0.98      0.98        45
   weighted avg       0.98      0.98      0.98        45



Finalizamos pegando o gabarito com as espécies (y_teste) para confrontar com os palpites que o modelo gerou (previsoes).

Na parte da matriz de confusão vai mostrar o cruzamento de erros e acertos usando o atributo '.T' para fazer uma trasposição de matriz pegando todas as linhas e transformando-as em colunas.

Já o relatório de classificação calcula as métricas: Precisão, Recall e F1-Score, para cada uma das três espécies e a média macro geral dessas métricas.